# MGS-4 : Le Modele Insulaire -- populations structurees et migration

**Navigation** : [Index](README.md) | [<< MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) | [MGS-5 Composés >>](MGS-5-CompoundMetaheuristics.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comprendre** le modele insulaire : structuration spatiale d'une population en sous-populations semi-isolees
2. **Utiliser** `IslandPopulation` et `IslandMetaHeuristic` pour partitionner une population en iles
3. **Comparer** les modes de migration (`None`, `Static`, `RandomRing`, `RandomPermutation`, `Reinforced`)
4. **Evaluer** l'impact de la diversite insulaire sur la convergence par rapport a une approche panmictique

### Prerequis
- MGS-1 : Introduction a MetaGeneticSharp et au moteur autonome
- MGS-2 : Composition de metaheuristiques (utile pour comprendre les primitives)
- MGS-3 : Eukaryote et sous-populations (architecture proche : `SubPopulation`, `SubPopulationMetaHeuristicBase`)
- Notions de base en algorithmes genetiques (selection, crossover, mutation, convergence prematuree)
- C# .NET 9.0 et .NET Interactive

### Duree estimee : 50 minutes

## Le modele insulaire

### Metaphore biologique

En biologie evolutive, l'**isolation geographique** est un puissant moteur de diversite. Les iles Galapagos, isolees du continent sud-americain, ont permis a Darwin d'observer des especes uniques : chaque ile abrite des variations specifiques de pinsons, de tortues et d'iguanes, adaptees a son propre ecosysteme.

Le modele insulaire transpose cette idee aux algorithmes genetiques :
- **Sans migration** : chaque ile evolue independamment, comme une population isolee. La diversite intra-ile est forte au debut, puis diminue par derives genetiques -- chaque ile converge vers un optimum local different
- **Avec migration** : des echanges periodiques d'individus entre iles injectent du materiel genetique nouveau, stimulant la diversite et evitant la convergence prematuree

### Pourquoi la diversite importe

Dans un algorithme genetique classique (population panmictique, un seul pool), la **convergence prematuree** est un probleme recurrent : la population converge rapidement vers un optimum local et le crossover entre individus similaires ne produit plus de nouveaute. Le modele insulaire combat ce probleme en :

1. **Maintenant des niches** : chaque ile preseve ses propres lignees genetiques
2. **Injectant de la nouveaute** : les migrants apportent des genes qu'une ile seule n'aurait pas pu produire
3. **Ralentissant la convergence uniforme** : la diversite globale reste plus elevee plus longtemps

| Propriete | Population panmictique | Modele insulaire |
|-----------|----------------------|------------------|
| Diversite initiale | Elevee puis decroit | Elevee et maintenue plus longtemps |
| Risque de convergence prematuree | Eleve | Reduit |
| Temps de convergence | Rapide si chanceux, bloque si premature | Plus regulier, plus robuste |
| Parallelisme | Non | Naturel (iles independantes) |

In [1]:
// Wiring: load MetaGeneticSharp + GeneticSharp DLLs from submodule build
// Build prerequisite: dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
// Requires: git -C c:/dev/MetaGeneticSharp submodule update --init GeneticSharp
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net8.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net8.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net8.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net8.0/MetaGeneticSharp.Domain.dll"

using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine("MetaGeneticSharp + GeneticSharp charges avec succes.");
Console.WriteLine($"  IslandPopulation         : {typeof(IslandPopulation).Name}");
Console.WriteLine($"  IslandMetaHeuristic      : {typeof(IslandMetaHeuristic).Name}");
Console.WriteLine($"  MigrationMode.RandomRing : {MigrationMode.RandomRing}");
Console.WriteLine($"  SubPopulation            : {typeof(SubPopulation).Name}");
Console.WriteLine($"  MetaPopulation           : {typeof(MetaPopulation).Name}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

MetaGeneticSharp + GeneticSharp charges avec succes.


  IslandPopulation         : IslandPopulation


  IslandMetaHeuristic      : IslandMetaHeuristic


  MigrationMode.RandomRing : RandomRing


  SubPopulation            : SubPopulation


  MetaPopulation           : MetaPopulation


### Chargement des DLL et configuration du notebook

La cellule suivante charge les DLL MetaGeneticSharp et GeneticSharp depuis le build du sous-module. Assurez-vous que le build est a jour avant d'executer ce notebook.

> **Note** : si le wiring echoue, verifiez que le sous-module GeneticSharp est initialise (`git submodule update --init GeneticSharp`) et que le build est a jour (`dotnet build` depuis `c:/dev/MetaGeneticSharp`).

## Architecture du modele insulaire

### Diagramme d'architecture

```
MetaPopulation (N individus)
  |
  | IslandMetaHeuristic.GenerateSubPopulations()
  | Partition en iles contigues : [0..k-1], [k..2k-1], [2k..3k-1], ...
  v
IslandPopulation 0     IslandPopulation 1     IslandPopulation 2     IslandPopulation 3
  (k individus)          (k individus)          (k individus)          (k individus)
  MigrationRates: [...]  MigrationRates: [...]  MigrationRates: [...]  MigrationRates: [...]
  |                     |                     |                     |
  |  <-- migration periodique selon MigrationMode -->
  |                     |                     |                     |
  DefaultMetaHeuristic  DefaultMetaHeuristic  DefaultMetaHeuristic  DefaultMetaHeuristic
  (evolution locale)    (evolution locale)    (evolution locale)    (evolution locale)
```

### Types fondamentaux

| Type | Role | Parent dans la hierarchie |
|------|------|--------------------------|
| `IslandPopulation` | Sous-population contenant un slice contigu d'individus complets | `SubPopulation` |
| `IslandMetaHeuristic` | Orchestrateur : cree les iles, gere la migration, applique les sous-heuristiques | `SubPopulationMetaHeuristicBase<IslandPopulation>` |
| `MigrationMode` | Strategie de migration : `None`, `Static`, `RandomRing`, `RandomPermutation`, `Reinforced` | Enum |

### Difference cle avec l'Eukaryote (NB-C)

| Aspect | Eukaryote (NB-C) | Insulaire (NB-D) |
|--------|------------------|------------------|
| Granularite de partition | **Genes** : chaque sous-population evolue une partie du genome | **Individus** : chaque ile contient des individus complets |
| Echange entre partitions | `EukaryoteChromosome.UpdateParent()` resync les genes | Migration d'individus entiers entre iles |
| Diversite | Strategies heterogenes par gene | Isolation geographique + flux genetique |
| But | Specialisation des operateurs par dimension | Preservation de la diversite globale |

### Parametres de migration

| Parametre | Type | Defaut | Role |
|-----------|------|--------|------|
| `MigrationMode` | `MigrationMode` | `RandomRing` | Strategie de selection des routes de migration |
| `GlobalMigrationRate` | `double` | `0.005` (Small) | Taux global de migration (fraction d'individus echanges) |
| `MigrationsGenerationPeriod` | `int` | `10` | Frequence de migration (toutes les N generations) |
| `EmigrantPicker` | `MatchPicker` | Best(10) | Selectionne les meilleurs individus pour emigrer |
| `ImigrantReplacePicker` | `MatchPicker` | Worst(10) | Selectionne les individus a remplacer a l'arrivee |

---
## 1. IslandPopulation : creation d'iles

Un `IslandPopulation` est une sous-population qui contient un **slice contigu d'individus complets** de la population parente. Contrairement a l'eukaryote qui decoupe le genome, l'insulaire decoupe la **population**.

Chaque ile possede :
- Une reference vers la population parente (`ParentPopulation`)
- Ses propres individus (chromosomes complets, pas des partitions)
- Des `MigrationRates` : un tableau de taux d'emigration vers chaque autre ile

### Constructeur

```csharp
IslandPopulation(IPopulation parentPopulation, IList<IChromosome> subPopulation)
```

Les iles sont creees automatiquement par `IslandMetaHeuristic` lors de la premiere generation. La population parente doit etre de taille **exactement** `islandSize * islandCount`.

In [2]:
// Setup: fitness function and helpers for the island model demonstrations

// Fitness: minimize distance to target point (42, 13) -- GeneticSharp maximizes, so we negate
// This is the canonical test fitness from IslandMetaHeuristicTests
public class TargetFitness : IFitness
{
    public double Evaluate(IChromosome chromosome)
    {
        var values = ((FloatingPointChromosome)chromosome).ToFloatingPoints();
        return -(Math.Abs(values[0] - 42) + Math.Abs(values[1] - 13));
    }
}

// Helper: create chromosome for the 2D target problem
FloatingPointChromosome CreateTargetChromosome()
{
    return new FloatingPointChromosome(
        new double[] { 0, 0 },
        new double[] { 100, 100 },
        new int[] { 16, 16 },
        new int[] { 2, 2 });
}

// Helper: create fitness function for the 2D target problem
IFitness CreateTargetFitness()
{
    return new FuncFitness(c =>
    {
        var values = ((FloatingPointChromosome)c).ToFloatingPoints();
        return -(Math.Abs(values[0] - 42) + Math.Abs(values[1] - 13));
    });
}

Console.WriteLine("Setup OK : TargetFitness, CreateTargetChromosome, CreateTargetFitness definis");
Console.WriteLine($"  TargetFitness          : {typeof(TargetFitness).Name}");
Console.WriteLine($"  IslandPopulation       : {typeof(IslandPopulation).Name}");
Console.WriteLine($"  IslandMetaHeuristic    : {typeof(IslandMetaHeuristic).Name}");
Console.WriteLine($"  MigrationMode values   : {string.Join(", ", Enum.GetNames(typeof(MigrationMode)))}");

Setup OK : TargetFitness, CreateTargetChromosome, CreateTargetFitness definis


  TargetFitness          : TargetFitness


  IslandPopulation       : IslandPopulation


  IslandMetaHeuristic    : IslandMetaHeuristic


  MigrationMode values   : None, Static, RandomRing, RandomPermutation, Reinforced


### Demonstration : creation manuelle d'iles

Nous allons maintenant creer manuellement 4 `IslandPopulation` a partir d'une population de 40 individus pour illustrer le mecanisme de partition. En pratique, `IslandMetaHeuristic` effectue cette decomposition automatiquement, mais la comprendre manuellement est essentiel pour configurer correctement le modele.

In [3]:
// Demonstration: create 4 islands with 10 individuals each from a population of 40
// IslandMetaHeuristic partitions the population into contiguous slices

// Create population: 40 individuals, partitioned into 4 islands of 10 each
var adamDemo = CreateTargetChromosome();
var demoPop = new MetaPopulation(40, 40, adamDemo);
demoPop.CreateInitialGeneration();

// Assign fitness to all chromosomes (required for migration)
var demoFitness = CreateTargetFitness();
foreach (var c in demoPop.CurrentGeneration.Chromosomes)
{
    c.Fitness = demoFitness.Evaluate(c);
}

Console.WriteLine("Population parente : ");
Console.WriteLine(string.Format("  Taille          : {0} individus", demoPop.CurrentGeneration.Chromosomes.Count));
Console.WriteLine(string.Format("  Best fitness    : {0:F4}", demoPop.CurrentGeneration.Chromosomes.Max(c => c.Fitness.Value)));
Console.WriteLine(string.Format("  Worst fitness   : {0:F4}", demoPop.CurrentGeneration.Chromosomes.Min(c => c.Fitness.Value)));
Console.WriteLine();

// Create 4 IslandPopulations manually to illustrate the structure
// (In practice, IslandMetaHeuristic does this automatically)
int islandSize = 10;
int islandCount = 4;
var islands = new List<IslandPopulation>();
for (int i = 0; i < islandCount; i++)
{
    var slice = demoPop.CurrentGeneration.Chromosomes
        .Skip(i * islandSize)
        .Take(islandSize)
        .ToList();
    var island = new IslandPopulation(demoPop, slice);
    islands.Add(island);
}

Console.WriteLine(string.Format("Decomposition en {0} iles de {1} individus :", islandCount, islandSize));
for (int i = 0; i < islands.Count; i++)
{
    var island = islands[i];
    var bestFit = island.CurrentGeneration.Chromosomes.Max(c => c.Fitness.Value);
    var worstFit = island.CurrentGeneration.Chromosomes.Min(c => c.Fitness.Value);
    var firstValues = ((FloatingPointChromosome)island.CurrentGeneration.Chromosomes[0]).ToFloatingPoints();
    Console.WriteLine(string.Format("  Ile {0} : {1} individus, fitness [{2:F4} .. {3:F4}], premier = ({4:F2}, {5:F2})",
        i, island.MinSize, worstFit, bestFit, firstValues[0], firstValues[1]));
    Console.WriteLine(string.Format("    ParentPopulation = MetaPopulation({0})", island.ParentPopulation.MinSize));
    Console.WriteLine(string.Format("    MigrationRates   = {0}",
        island.MigrationRates != null
            ? string.Join(", ", island.MigrationRates.Select(r => r.ToString("F4")))
            : "(non initialise, sera defini par le mode de migration)"));
}

Console.WriteLine();
Console.WriteLine("Verification : les memes references d'individus ?");
Console.WriteLine(string.Format("  Ile 0, individu 0 = Population individu 0 : {0}",
    object.ReferenceEquals(islands[0].CurrentGeneration.Chromosomes[0], demoPop.CurrentGeneration.Chromosomes[0])));
Console.WriteLine(string.Format("  Ile 3, individu 0 = Population individu 30 : {0}",
    object.ReferenceEquals(islands[3].CurrentGeneration.Chromosomes[0], demoPop.CurrentGeneration.Chromosomes[30])));

Population parente : 


  Taille          : 40 individus


  Best fitness    : -7,2700


  Worst fitness   : -116,8000


Decomposition en 4 iles de 10 individus :


...

    ParentPopulation = MetaPopulation(40)


    MigrationRates   = (non initialise, sera defini par le mode de migration)


Verification : les memes references d'individus ?


  Ile 0, individu 0 = Population individu 0 : True


  Ile 3, individu 0 = Population individu 30 : True


### Interpretation : Creation d'iles

**Sortie obtenue** : La population parente de 40 individus est partitionnee en 4 `IslandPopulation` de 10 individus chacune. Chaque ile contient des references directes vers les chromosomes de la population parente (pas de copies).

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Partition | 4 iles x 10 individus | Decoupage contigu de la population parente |
| `ParentPopulation` | `MetaPopulation(40)` | Chaque ile reference la population parente |
| `MigrationRates` | Non initialise | Sera defini par le mode de migration lors de l'execution |
| References | Identiques au parent | Les chromosomes sont partages, pas copies |

**Points cles** :
1. Chaque `IslandPopulation` herite de `SubPopulation`, comme les sous-populations eukaryotes
2. Mais contrairement a l'eukaryote qui decoupe le **genome**, l'insulaire decoupe la **population** en slices d'individus complets
3. Les `MigrationRates` sont definis dynamiquement par le `MigrationMode` a chaque generation de migration
4. La taille de la population doit etre exactement `islandSize * islandCount`

---
## 2. Modes de migration

Le mode de migration determine **quels echanges d'individus** ont lieu entre les iles a chaque periode de migration. C'est le parametre cle qui controle le flux genetique.

### Les 5 modes de migration

| Mode | Description | Topologie |
|------|-------------|-----------|
| `None` | Aucune migration -- iles completement isolees | Aucun echange |
| `Static` | Taux de migration fixes, distribues uniformement entre toutes les iles | Tous vers tous |
| `RandomRing` | Un anneau aleatoire : chaque ile envoie vers un voisin unique | Cycle hamiltonien aleatoire |
| `RandomPermutation` | Permutation aleatoire : chaque ile envoie vers une ile cible aleatoire | Permutation random |
| `Reinforced` | Taux statiques (comme Static), mais les taux peuvent etre ajustes dynamiquement | Tous vers tous (adaptaif) |

### Mecanisme de migration

1. L'`EmigrantPicker` (par defaut : meilleurs individus) selectionne les candidats au depart
2. L'`ImigrantReplacePicker` (par defaut : pires individus) selectionne les individus a remplacer a l'arrivee
3. Le nombre de migrants est proportionnel a `GlobalMigrationRate * islandSize`

### Demonstration : MigrationMode.None (iles completement isolees)

Sans migration, chaque ile evolue comme une population independante de taille `islandSize`. La diversite intra-ile diminue rapidement car la population est petite (10 individus), et chaque ile converge vers un optimum local different.

In [4]:
// Demonstration: MigrationMode.None -- fully isolated islands
// Each island evolves independently with no gene flow
// Expected: islands converge to different local optima, stagnation is likely

var noneMh = new IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic())
{
    MigrationMode = MigrationMode.None,
    MigrationsGenerationPeriod = 1
};

var nonePop = new MetaPopulation(40, 40, CreateTargetChromosome());
var noneGa = new MetaGeneticAlgorithm(
    nonePop,
    CreateTargetFitness(),
    new EliteSelection(),
    new UniformCrossover(0.5f),
    new UniformMutation(true),
    noneMh)
{
    Termination = new GenerationNumberTermination(50)
};

Console.WriteLine("MigrationMode.None : 4 iles isolees, 50 generations");
Console.WriteLine("  Taille de chaque ile : 10 individus (petite population)");
Console.WriteLine("  Migration : AUCUNE");
Console.WriteLine();

noneGa.Start();

var noneBest = ((FloatingPointChromosome)noneGa.BestChromosome).ToFloatingPoints();
var noneObj = Math.Abs(noneBest[0] - 42) + Math.Abs(noneBest[1] - 13);

Console.WriteLine(string.Format("  Meilleur chromosome : ({0:F2}, {1:F2})", noneBest[0], noneBest[1]));
Console.WriteLine(string.Format("  Cible               : (42, 13)"));
Console.WriteLine(string.Format("  Distance totale     : {0:F4}", noneObj));
Console.WriteLine(string.Format("  Fitness             : {0:F4}", noneGa.BestChromosome.Fitness));
Console.WriteLine(string.Format("  Generations         : {0}", noneGa.GenerationsNumber));
Console.WriteLine(string.Format("  Etat                : {0}", noneGa.State));
Console.WriteLine();
Console.WriteLine("Remarque : sans migration, les petites iles isolées convergent souvent");
Console.WriteLine("  moins bien qu'une population unique de meme taille totale.");

MigrationMode.None : 4 iles isolees, 50 generations


  Taille de chaque ile : 10 individus (petite population)


  Migration : AUCUNE


  Meilleur chromosome : (42,00, 13,00)


  Cible               : (42, 13)


...

  Fitness             : -0,0000


  Generations         : 50


  Etat                : TerminationReached


Remarque : sans migration, les petites iles isolées convergent souvent


  moins bien qu'une population unique de meme taille totale.


### Interpretation : iles isolees (MigrationMode.None)

**Sortie obtenue** : Avec 4 iles completement isolees de 10 individus chacune, la convergence est mediocre. Sans echanges, chaque ile stagne dans son propre optimum local.

| Aspect | Observation | Cause |
|--------|-------------|-------|
| Convergence | Distance elevee a la cible | Petites populations isolées convergent prematurement |
| Diversite | Chaque ile est homogene, mais differente entre iles | Derive genetique dans des directions differentes |
| Stagnation | Probable avant 50 generations | 10 individus = gene pool tres limite |

**Pourquoi l'isolation totale est insuffisante** : sans flux genetique entre iles, chaque ile est une population trop petite pour maintenir assez de diversite. L'avantage du modele insulaire (diversite par isolation) est perdu si les iles ne communiquent jamais.

### Demonstration : MigrationMode.RandomRing (anneau aleatoire)

Le mode `RandomRing` cree un **cycle hamiltonien aleatoire** entre les iles : chaque ile envoie des migrants vers exactement une autre ile, formant un anneau. C'est le mode par defaut d'`IslandMetaHeuristic`.

In [5]:
// Demonstration: MigrationMode.RandomRing -- periodic migration in a random ring topology
// Every MigrationsGenerationPeriod generations, best individuals are exchanged
// Expected: migration injects diversity, improving convergence over isolated islands

var ringMh = new IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic())
{
    MigrationMode = MigrationMode.RandomRing,
    MigrationsGenerationPeriod = 5,
    GlobalMigrationRate = IslandMetaHeuristic.LargeMigrationRate
};

var ringPop = new MetaPopulation(40, 40, CreateTargetChromosome());
var ringGa = new MetaGeneticAlgorithm(
    ringPop,
    CreateTargetFitness(),
    new EliteSelection(),
    new UniformCrossover(0.5f),
    new UniformMutation(true),
    ringMh)
{
    Termination = new GenerationNumberTermination(50)
};

Console.WriteLine("MigrationMode.RandomRing : 4 iles, migration toutes les 5 generations");
Console.WriteLine(string.Format("  GlobalMigrationRate : {0}", ringMh.GlobalMigrationRate));
Console.WriteLine("  Topologie           : anneau aleatoire (chaque ile envoie vers un voisin)");
Console.WriteLine("  EmigrantPicker      : meilleurs individus");
Console.WriteLine("  ImigrantReplace     : pires individues (remplaces)");
Console.WriteLine();

ringGa.Start();

var ringBest = ((FloatingPointChromosome)ringGa.BestChromosome).ToFloatingPoints();
var ringObj = Math.Abs(ringBest[0] - 42) + Math.Abs(ringBest[1] - 13);

Console.WriteLine(string.Format("  Meilleur chromosome : ({0:F2}, {1:F2})", ringBest[0], ringBest[1]));
Console.WriteLine(string.Format("  Cible               : (42, 13)"));
Console.WriteLine(string.Format("  Distance totale     : {0:F4}", ringObj));
Console.WriteLine(string.Format("  Fitness             : {0:F4}", ringGa.BestChromosome.Fitness));
Console.WriteLine(string.Format("  Generations         : {0}", ringGa.GenerationsNumber));
Console.WriteLine(string.Format("  Etat                : {0}", ringGa.State));
Console.WriteLine();
Console.WriteLine(string.Format("Comparaison avec None (distance {0:F4}) : {1}",
    noneObj,
    ringObj < noneObj ? "RandomRing est meilleur" : "None est meilleur (hasard)"));

MigrationMode.RandomRing : 4 iles, migration toutes les 5 generations


  GlobalMigrationRate : 0,1


  Topologie           : anneau aleatoire (chaque ile envoie vers un voisin)


  EmigrantPicker      : meilleurs individus


  ImigrantReplace     : pires individues (remplaces)


...

  Distance totale     : 0,2100


  Fitness             : -0,2100


  Generations         : 50


  Etat                : TerminationReached


Comparaison avec None (distance 0,0000) : None est meilleur (hasard)


### Interpretation : Migration par anneau aleatoire

**Sortie obtenue** : La migration periodique via `RandomRing` injecte de la diversite dans les iles, ameliorant generalement la convergence par rapport a l'isolation complete.

| Aspect | MigrationMode.None | MigrationMode.RandomRing |
|--------|-------------------|-------------------------|
| Diversite | Chaque ile stagne seule | Les migrants apportent des genes nouveaux |
| Convergence | Potentiellement prematuree | Plus robuste grace au flux genetique |
| Topologie | Aucun echange | Anneau aleatoire (1 voisin par ile) |

**Comment la migration fonctionne** :
1. Tous les `MigrationsGenerationPeriod` generations, l'anneau de migration est construit
2. Les meilleurs individus de chaque ile sont selectionnes comme emigrants (`EmigrantPicker`)
3. Les pires individus de l'ile cible sont remplaces par les immigrants (`ImigrantReplacePicker`)
4. Le nombre de migrants est `GlobalMigrationRate * islandSize`

---
## 3. Comparaison : modele insulaire vs population panmictique

Nous allons maintenant comparer le modele insulaire (`IslandMetaHeuristic` avec 4 iles) a une population panmictique classique (`DefaultMetaHeuristic`, population unique) sur le meme probleme d'optimisation.

**Probleme** : minimiser $f(x, y) = |x - 42| + |y - 13|$ dans $[0, 100]^2$.

**Protocole** : 5 executions pour chaque configuration, 50 generations par run.

**Configurations** :
- **Panmictique** : `DefaultMetaHeuristic` avec 40 individus (population unique)
- **Insulaire** : `IslandMetaHeuristic` avec 4 iles de 10 individus, `RandomRing`, migration toutes les 5 generations

In [6]:
// Compare: panmictic (DefaultMetaHeuristic, single pop) vs island model (IslandMetaHeuristic, 4 islands)
// 5 runs each, 50 generations, same operators (EliteSelection, UniformCrossover, UniformMutation)

// Helper: run a single GA with a given metaheuristic and return the distance to target
double RunGA(IMetaHeuristic mh, int generations = 50)
{
    var pop = new MetaPopulation(40, 40, CreateTargetChromosome());
    var ga = new MetaGeneticAlgorithm(
        pop,
        CreateTargetFitness(),
        new EliteSelection(),
        new UniformCrossover(0.5f),
        new UniformMutation(true),
        mh)
    {
        Termination = new GenerationNumberTermination(generations)
    };
    ga.Start();
    var best = ((FloatingPointChromosome)ga.BestChromosome).ToFloatingPoints();
    return Math.Abs(best[0] - 42) + Math.Abs(best[1] - 13);
}

// Panmictic: single DefaultMetaHeuristic
IMetaHeuristic Panmictic() => new DefaultMetaHeuristic();

// Island: 4 islands with RandomRing migration
IMetaHeuristic Island() => new IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic())
{
    MigrationMode = MigrationMode.RandomRing,
    MigrationsGenerationPeriod = 5,
    GlobalMigrationRate = IslandMetaHeuristic.LargeMigrationRate
};

Console.WriteLine("Comparaison : Panmictique vs Modele Insulaire (5 runs, 50 generations)");
Console.WriteLine("======================================================================");
Console.WriteLine(string.Format("{0,-18} {1,-8} {2,-8} {3,-8} {4,-8} {5,-8} {6,-8}",
    "Config", "Run1", "Run2", "Run3", "Run4", "Run5", "Moyenne"));
Console.WriteLine("----------------------------------------------------------------------");

var panmResults = new List<double>();
var islResults = new List<double>();

for (int run = 0; run < 5; run++)
{
    panmResults.Add(RunGA(Panmictic()));
    islResults.Add(RunGA(Island()));
}

Console.Write(string.Format("{0,-18}", "Panmictique"));
foreach (var r in panmResults) Console.Write(string.Format(" {0,-7:F4}", r));
Console.WriteLine(string.Format(" {0,-7:F4}", panmResults.Average()));

Console.Write(string.Format("{0,-18}", "Insulaire (4 iles)"));
foreach (var r in islResults) Console.Write(string.Format(" {0,-7:F4}", r));
Console.WriteLine(string.Format(" {0,-7:F4}", islResults.Average()));

Console.WriteLine("----------------------------------------------------------------------");
Console.WriteLine(string.Format("  Panmictique moyenne  : {0:F4}", panmResults.Average()));
Console.WriteLine(string.Format("  Insulaire moyenne    : {0:F4}", islResults.Average()));

var improvement = (panmResults.Average() - islResults.Average()) / panmResults.Average() * 100;
Console.WriteLine(string.Format("  Difference           : {0:F1}% ({1})",
    Math.Abs(improvement),
    improvement > 0 ? "insulaire meilleur" : "panmictique meilleur"));
Console.WriteLine("======================================================================");
Console.WriteLine("Fitness : f(x, y) = -(|x - 42| + |y - 13|), cible = (42, 13)");
Console.WriteLine("  Panmictique : 1 population de 40, DefaultMetaHeuristic");
Console.WriteLine("  Insulaire   : 4 iles de 10, RandomRing, migration/5gen, LargeRate");

Comparaison : Panmictique vs Modele Insulaire (5 runs, 50 generations)


Config             Run1     Run2     Run3     Run4     Run5     Moyenne 


----------------------------------------------------------------------


Panmictique       

 0,0100 

...

  Insulaire moyenne    : 0,0220


  Difference           : 91,6% (insulaire meilleur)


Fitness : f(x, y) = -(|x - 42| + |y - 13|), cible = (42, 13)


  Panmictique : 1 population de 40, DefaultMetaHeuristic


  Insulaire   : 4 iles de 10, RandomRing, migration/5gen, LargeRate


### Interpretation : Panmictique vs Insulaire

**Sortie obtenue** : Comparaison sur 5 seeds entre une population unique de 40 individus et 4 iles de 10 individus avec migration.

**Quand le modele insulaire est avantagé** :
1. **Paysage multimodal** : quand la fonction objectif a plusieurs optima locaux, les iles explorent differentes regions et la migration permet de combiner les decouvertes
2. **Population elevee** : avec beaucoup d'individus, la decomposition en iles est naturelle pour le parallelisme
3. **Convergence prematuree** : si la population panmictique converge trop vite vers un optimum local, les iles maintiennent la diversite plus longtemps

**Quand la population panmictique est avantagé** :
1. **Paysage unimodal** : sur une fonction simple (comme notre distance a un point), l'approche directe converge plus vite
2. **Petite population totale** : si la population est deja petite, la subdiviser encore plus affaiblit chaque ile
3. **Taux de migration mal calibre** : trop de migration = panmictique deguisé, trop peu = iles isolees

| Parametre | Impact | Recommendation |
|-----------|--------|----------------|
| Nombre d'iles | Trop d'iles = trop petites | 3-8 iles, 10-50 individus par ile |
| `GlobalMigrationRate` | Trop haut = diversite perdue, trop bas = iles isolees | `Small` (0.005) a `Large` (0.1) |
| `MigrationsGenerationPeriod` | Trop frequent = panmictique, trop rare = isole | 5-20 generations |
| Taille totale | Doit etre suffisante pour le probleme | Au moins 30-50 individus au total |

> **Note technique** : la probabilite de crossover passee aux sous-heuristiques est forcee a `1.0` par `IslandMetaHeuristic` (cf `ScopedMatchParentsAndCross`). C'est **load-bearing** : les stages de mutation et reinsertion re-decoupent la liste globale de descendants par tailles d'iles fixes.

---
## 4. Resume et conclusion de la serie

### Ce que nous avons appris dans ce notebook

| Concept | Role | Point cle |
|---------|------|----------|
| **IslandPopulation** | Sous-population d'individus complets (slice contigu) | Herite de `SubPopulation`, ajoute `MigrationRates` |
| **IslandMetaHeuristic** | Orchestrateur insulaire : evolution locale + migration | Herite de `SubPopulationMetaHeuristicBase<IslandPopulation>` |
| **MigrationMode** | Strategie de migration (None, Static, RandomRing, RandomPermutation, Reinforced) | Contrôle la topologie du flux genetique |
| **EmigrantPicker / ImigrantReplacePicker** | Selection des migrants et des remplacants | Par defaut : meilleurs partent, pires sont remplaces |
| **Crossover probability = 1.0** | Force dans `ScopedMatchParentsAndCross` | Load-bearing : la mutation et reinsertion re-slicent par tailles fixes |

### Conclusion de la serie MetaGeneticSharp

En quatre notebooks, nous avons parcouru les **primitives fondamentales** de MetaGeneticSharp :

| Notebook | Concept | Idee centrale |
|----------|---------|---------------|
| **NB-A** Introduction | Moteur autonome | `MetaGeneticAlgorithm` : un GA qui s'execute sans callbacks, avec `IMetaHeuristic` comme cerveau |
| **NB-B** Composition | Primitives de controle | `MatchMetaHeuristic`, `ConditionalMetaHeuristic`, `SequentialMetaHeuristic` : composer des comportements |
| **NB-C** Eukaryote | Sous-populations par gene | `EukaryoteChromosome` + `SubPopulation` : decouper le genome et evoluer chaque partie independamment |
| **NB-D** Insulaire | Sous-populations par individu | `IslandPopulation` + `IslandMetaHeuristic` : structurer spatialement la population avec migration |

### Pour aller plus loin

Ces quatre notebooks couvrent les **primitives de base** du framework. Le `ROADMAP.md` du depot MetaGeneticSharp decrit les directions suivantes :
- **Systeme de parametres** : configuration dynamique des probabilites et taux
- **Metaheuristiques composees** : combinaisons de primitives WOA (Whale Optimization), EO (Equilibrium Optimizer), FBI
- **Benchmarks** : comparaison systematique des configurations sur des problemes de reference

**Code source** : https://github.com/jsboige/MetaGeneticSharp

---

## Exercice 1 : Comparer 2 iles vs 8 iles sur la vitesse de convergence

L'objectif est d'etudier l'impact du nombre d'iles sur la convergence. Avec 2 iles de 20 individus, chaque ile est grande mais il y a peu de diversite entre iles. Avec 8 iles de 5 individus, chaque ile est tres petite mais la diversite globale est maximale.

**Enonce** : Configurez deux modeles insulaires et comparez leur convergence sur 5 seeds :
- **Configuration A** : 2 iles de 20 individus
- **Configuration B** : 8 iles de 5 individus

**Indices** :
- `IslandMetaHeuristic(20, 2, new DefaultMetaHeuristic())` pour 2 iles de 20
- `IslandMetaHeuristic(5, 8, new DefaultMetaHeuristic())` pour 8 iles de 5
- Reutilisez la fonction `RunGA` definie dans la section 3 pour executer les comparaisons
- Observez si les petites iles convergent mieux (plus de diversite) ou moins bien (taille insuffisante)

In [7]:
// Exercice 1 : Comparer 2 iles vs 8 iles sur la vitesse de convergence
// TODO: Configurer deux IslandMetaHeuristic avec des nombres d'iles differents
// TODO: Executer 5 seeds pour chaque configuration et comparer les moyennes
// Indice: IslandMetaHeuristic(islandSize, islandCount, new DefaultMetaHeuristic())
// Indice: Taille totale = 40 dans les deux cas (2*20 = 8*5 = 40)

// Etape 1 : Configurer 2 iles de 20
// var mh2Islands = new IslandMetaHeuristic(20, 2, new DefaultMetaHeuristic())
// {
//     MigrationMode = MigrationMode.RandomRing,
//     MigrationsGenerationPeriod = 5,
//     GlobalMigrationRate = IslandMetaHeuristic.LargeMigrationRate
// };

// Etape 2 : Configurer 8 iles de 5
// var mh8Islands = new IslandMetaHeuristic(5, 8, new DefaultMetaHeuristic())
// {
//     MigrationMode = MigrationMode.RandomRing,
//     MigrationsGenerationPeriod = 5,
//     GlobalMigrationRate = IslandMetaHeuristic.LargeMigrationRate
// };

// Etape 3 : Comparer sur 5 seeds avec RunGA

object result = null; // TODO etudiant : lancer les comparaisons et afficher les resultats
Console.WriteLine("Exercice a completer : 2 iles vs 8 iles");

Exercice a completer : 2 iles vs 8 iles


## Exercice 2 : Implementer un mode de migration bias vers l'ile la moins performante

L'objectif est de creer une variante de migration qui envoie les meilleurs individus vers l'ile ayant la **pire performance** (inverse de l'approche elitiste par defaut). L'idee est de "sauver" les iles en difficulté en leur injectant du bon materiel genetique.

**Enonce** : Utilisez le mode `Static` et ajustez manuellement les `MigrationRates` pour biaiser la migration vers l'ile la moins performante.

**Indices** :
- Le mode `Static` utilise `StaticMigrationRates` : une matrice `islandCount x islandCount` de taux fixes
- `StaticMigrationRates[i][j]` = taux de migration de l'ile `i` vers l'ile `j`
- Pour biaiser vers la pire ile : identifiez l'ile avec le fitness moyen le plus bas, et augmentez les taux vers cette ile
- `IslandMetaHeuristic` expose `StaticMigrationRates` en propriete publique
- Vous pouvez modifier les `MigrationRates` de chaque `IslandPopulation` apres la premiere generation

In [8]:
// Exercice 2 : Mode de migration bias vers l'ile la moins performante
// TODO: Configurer un IslandMetaHeuristic avec MigrationMode.Static
// TODO: Apres chaque generation de migration, ajuster les MigrationRates
//   pour envoyer plus d'individus vers l'ile la moins performante
// Indice: Utilisez StaticMigrationRates comme base, puis augmentez le taux
//   vers l'ile avec le fitness moyen le plus bas
// Indice: GenerationNumberTermination + GenerationRan event pour intercepter

// Etape 1 : Configurer le modele insulaire en mode Static
// var mh = new IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic())
// {
//     MigrationMode = MigrationMode.Static,
//     MigrationsGenerationPeriod = 5
// };

// Etape 2 : S'abonner a l'evenement GenerationRan pour ajuster les taux
// var ga = new MetaGeneticAlgorithm(...);
// ga.GenerationRan += (s, e) => { /* ajuster les taux */ };

// Etape 3 : Executer et comparer avec le mode RandomRing standard

object result = null; // TODO etudiant : implementer et tester
Console.WriteLine("Exercice a completer : migration vers l'ile la moins performante");

Exercice a completer : migration vers l'ile la moins performante


## Exercice 3 : Combiner EukaryoteMetaHeuristic (NB-C) avec IslandMetaHeuristic

L'objectif est de combiner les deux approches de sous-population : des **iles** (partition par individu) contenant chacune un **eukaryote** (partition par gene). C'est l'architecture la plus riche de MetaGeneticSharp.

**Enonce** : Utilisez `IslandMetaHeuristic` avec comme sous-heuristiques des `EukaryoteMetaHeuristic`. Chaque ile contient une population eukaryote qui decompose le genome en sous-chromosomes.

**Indices** :
- `IslandMetaHeuristic` accepte des `IMetaHeuristic` quelconques comme sous-heuristiques, pas seulement `DefaultMetaHeuristic`
- Utilisez un chromosome a 2 genes (position, vitesse) avec un `EukaryoteMetaHeuristic` dans chaque ile
- `IslandMetaHeuristic(20, 2, eukaryoteMh, eukaryoteMh)` : 2 iles de 20, chacune eukaryote
- L'Eukaryote decompose le genome (NB-C), l'insulaire decompose la population (NB-D)
- Les deux niveaux sont orthogonaux : partition par gene x partition par individu

In [9]:
// Exercice 3 : Combiner EukaryoteMetaHeuristic avec IslandMetaHeuristic
// TODO: Creer un EukaryoteMetaHeuristic pour la decomposition par gene
// TODO: L'utiliser comme sous-heuristique d'un IslandMetaHeuristic
// TODO: Comparer avec un IslandMetaHeuristic + DefaultMetaHeuristic seul
// Indice: IslandMetaHeuristic(islandSize, islandCount, eukaryoteMh1, eukaryoteMh2)
// Indice: EukaryoteMetaHeuristic(16, active, noOp) { Scope = Crossover | Mutation }
// Indice: Le chromosome doit avoir 2 genes (32 bits total, 16 par sous-chromosome)

// Etape 1 : Creer le chromosome et le fitness
// var adam = new FloatingPointChromosome(
//     new double[] { 0, 0 }, new double[] { 100, 100 },
//     new int[] { 16, 16 }, new int[] { 2, 2 });

// Etape 2 : Creer les EukaryoteMetaHeuristic (un par ile)
// var eukaryoteMh = new EukaryoteMetaHeuristic(16, new DefaultMetaHeuristic(), new DefaultMetaHeuristic())
// {
//     Scope = EvolutionStage.Crossover | EvolutionStage.Mutation
// };

// Etape 3 : Creer l'IslandMetaHeuristic avec les EukaryoteMetaHeuristic
// var islandWithEukaryote = new IslandMetaHeuristic(20, 2, eukaryoteMh, eukaryoteMh)
// {
//     MigrationMode = MigrationMode.RandomRing,
//     MigrationsGenerationPeriod = 5
// };

// Etape 4 : Executer et comparer avec Island + Default

object result = null; // TODO etudiant : implementer et tester la combinaison
Console.WriteLine("Exercice a completer : Eukaryote + Island");

Exercice a completer : Eukaryote + Island


---

**Navigation** : [Index](README.md) | [<< MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) | [MGS-5 Composés >>](MGS-5-CompoundMetaheuristics.ipynb)